<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/ai_ball_steps_bounce_arg_patch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ai_ball_steps_directions_patch.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/17uAhylnn5dFjtkP1yCzd51NiHChu83OC

ai_ball_7.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1RScD6nl_1_teIc-eGZ_Tao-A5BP6xo1J

# AI Ball — Ball Trap Brain

A toy "ball trap" symbolic model: input characters are tokenized against an alphabet, turned into a ball trajectory through a ring of deflectors, and the deflectors the ball falls through ("holes") produce the output characters.

This notebook contains:
1. The tokenizer and the `Ball_Trap_Brain` model itself
2. Fitness evaluation utilities
3. A genetic algorithm (`main`) that evolves a single brain
4. A `Council_Brain` that combines several independently-trained brains by majority vote
5. A **continuous relaxation** training method (`Ball_Trap_Brain.train_continuous`) that trains a brain with gradient descent instead of the genetic algorithm, and can optionally keep the trained continuous parameters usable directly (`predict_continuous`, `evaluate_continuous`) instead of only ever discretizing them back into the original discrete brain
6. An optional, guarded cell that reproduces the original end-to-end council training run

## Imports & tokenizer


In [ ]:
import random
import math
import copy

In [ ]:
def TokenizeInput(input, alphabet):
    input_str = str(input)
    matches = []

    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue

        # Loop to find ALL occurrences of the item instead of just the first
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break

            matches.append((index, -len(item_str), order, item))

            # Move start position forward by 1 to continue searching the rest of the string
            # (Adding 1 instead of len(item_str) allows it to safely catch overlapping patterns)
            start_pos = index + 1

    # Sorts by starting index, then by longest string, then by original order
    matches.sort(key=lambda match: (match[0], match[1], match[2]))

    non_overlapping_items = []
    last_end_index = 0

    for match in matches:
        start_index = match[0]
        length = -match[1]  # The length was stored as a negative number
        item = match[3]

        # Only add the token if it doesn't overlap with the previously added token
        if start_index >= last_end_index:
            non_overlapping_items.append(item)
            last_end_index = start_index + length
    #print(non_overlapping_items)
    return non_overlapping_items

## `Ball_Trap_Brain`

Includes the original genetic-algorithm-oriented methods (`clone`, `Mutate`, `shift_alphabet`, ...) plus a gradient-based continuous relaxation trainer, split across a few methods:
- `train_continuous(...)`: trains the continuous parameters with gradient descent (see its docstring for details on the relaxation scheme). Temperature cools from `temp_start` to `temp_end` (linear) and `index_temp` cools from `index_temp_start` to `index_temp_end` (power-`index_temp_power` schedule) over `epochs`, with the learning rate cosine-decaying alongside them and gradients clipped each step for stability; by default the best-loss checkpoint seen during training is restored at the end rather than whatever the final epoch produced. Returns `(loss_history, accuracy_history)`.
- `predict_continuous(input_str, ...)`: answer a question using the **continuous** brain directly, without discretizing anything -- no dependency on `deflectors_list` etc at all.
- `evaluate_continuous(test_cases, ...)`: like `evaluate_performance`, but scores `predict_continuous` instead of `compute_input`.
- `commit_continuous()`: discretize already-trained continuous parameters into the discrete brain without training any further.

**Two pollution bugs, now fixed, that affected discrete *and* continuous output alike:**
1. *Leading-position leakage.* The loss only scored a right-aligned tail of each trajectory; leading positions outside that window had no training signal at all and could leak arbitrary symbols into the real output. Fixed by folding "should this position emit at all" into the *same* per-position loss as a genuine "emit nothing" class -- `never_fired = 1 - out_dist.sum()` already forms a valid (K+1)-way distribution with the K real symbols, so every position (aligned or not) now has a well-defined target under one unified objective, instead of a second, separately-weighted penalty term that destabilized training by fighting the main loss over shared deflectors.
2. *Repeat/self-loop cycling.* Once a trajectory hit a hole, the "first hole reached" objective stopped caring what happened for the rest of its `cap` steps -- so a hole deflector could cycle with another hole (or even itself) and just re-emit symbols for the whole `cap` budget (e.g. `"3333333333"`, `"2323232323"`). Fixed with `repeat_penalty_weight` (default 0.3), which penalizes the expected number of *additional* hole-hits after the first one per trajectory -- computed from the same per-example forward pass as the main loss, so it targets whatever cycle each specific trajectory is actually at risk of.

**`commit` flag:** `train_continuous(..., commit=True)` (the default) discretizes the trained parameters back into this instance's own `deflectors_list`, `alphabet_directions` and `alphabet_start_positions` -- so every other method (`compute_input`, `ricochet`, `clone`, `Mutate`, ...) keeps working exactly as before, just with gradient-trained values. Pass `commit=False` to leave the discrete brain completely untouched and only keep the trained continuous version around for `predict_continuous`/`evaluate_continuous` -- useful for comparing continuous vs. discrete behavior, or for iterating with more `train_continuous` calls before ever committing to a discrete brain (call `commit_continuous()` whenever you're ready to).

**How `token_acc` is scored (printed during training):** within each epoch's batch, for every trajectory position (aligned *or* blank), it checks whether the predicted class -- top symbol if emit-probability >= 0.5, else "blank" -- matches the target class, and reports the fraction that do. It's read off the same numbers already used for the loss -- no extra simulation -- but it's still a *character-position* metric, not a *question* metric: it doesn't check whether a whole answer matches end to end, and it's computed on the soft relaxation, not the real discrete brain. For real, whole-question exact-match accuracy, use `evaluate_performance` (discrete) or `evaluate_continuous` (continuous).


In [ ]:
class Ball_Trap_Brain:
    def __init__(self, deflectors, bounce, alphabet, output, holes_multiplier=1, direction_range=3, max_bounces=10):
        self.output = output
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce*2
        self.alphabet = alphabet
        # The maximum number of deflector hits ("bounces") a single trajectory
        # is allowed before ricochet()/ricochet_deflectors() cut it off, even if
        # ball_energy hasn't run out yet. Previously hardcoded to 10 at every
        # call site (compute_input, compute_deflectors, and the continuous
        # relaxation's cap metadata) -- now a single configurable value.
        self.max_bounces = max_bounces
        # Each alphabet token gets its own integer direction in
        # [-direction_range, direction_range] instead of only +-1. A wider range
        # gives travel()'s leave-one-out, position-weighted sum far more
        # resolution to separate tokens by -- with only +-1, two different tokens
        # were indistinguishable to the sum beyond their single bit of sign, which
        # was a capacity bottleneck independent of (and compounding) the
        # order-invariance one that position weighting alone fixes.
        self.direction_range = direction_range
        self.alphabet_directions = self.choose_alphabet_direction(len(alphabet))
        self.alphabet_start_positions = self.start_positions(len(alphabet), deflectors)
        self.deflectors_list = [[random.randint(0, deflectors-1), bounce] for i in range(deflectors)]
        self.set_holes(output, deflectors, self.holes_multiplier)

    def set_holes (self, output, deflectors, holes_multiplier):
        for i in range(len(output)):
          for j in range(holes_multiplier):
            self.deflectors_list[random.randint(0, deflectors-1)][1] = output[i]

    def choose_alphabet_direction(self, alphabet):
        return [random.randint(-self.direction_range, self.direction_range) for _ in range(alphabet)]

    def start_positions(self, alphabet, deflectors):
        return [random.randint(0, deflectors-1) for i in range(alphabet)]


    def compute_input(self, input):
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))


        return self.ricochet(travel, self.max_bounces, self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        output = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    output.append(val)
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return output

    def compute_deflectors(self, input):
        """
        Processes input exactly like compute_input but returns an array
        of all deflectors hit during the ball's travel.
        """
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))

        # Passes the configured max_bounces cap and the current ball_energy
        return self.ricochet_deflectors(travel, self.max_bounces, self.ball_energy)

    def ricochet_deflectors(self, travel, cap, ball_energy):
        """
        Modified ricochet logic that records and returns the deflectors
        hit instead of the output values.
        """
        deflectors_hit = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]

                # Append the deflector's details to our tracking array
                deflectors_hit.append(new_idx)

                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return deflectors_hit

    def travel(self, directions, position, index):
        # Linear position weighting: weight token j's direction by (j+1) instead of
        # a flat, order-invariant sum. This means swapping two positions changes the
        # total (unless their directions happen to be equal), breaking the old
        # collision where every anagram of the same characters produced the exact
        # same travel value -- and therefore the exact same output -- regardless of
        # order. Skip index-in-place rather than pop()+re-sum, so the weights (j+1)
        # still refer to true sequence positions rather than shifted ones.
        #
        # Normalized by the sum of weights actually used (i.e. excluding `index`)
        # instead of left as a raw sum. With a raw sum, the weight (j+1) grows with
        # sequence length, so `total`'s typical magnitude grows roughly quadratically
        # with the number of tokens (worst case exactly quadratic, when directions are
        # all the same sign) -- for long inputs this blows `total` up to many multiples
        # of len(deflectors_list), so the `% n` in ricochet() effectively randomizes the
        # starting deflector regardless of actual token content, and the raw, un-wrapped
        # `distance` hop at that near-random start can drain most of ball_energy on the
        # very first step. Both effects make long sequences increasingly likely to never
        # reach a hole at all, collapsing the output down to only the few trajectories
        # that get lucky -- often just one. Dividing by the weight sum keeps `total`'s
        # magnitude bounded to roughly [-direction_range, direction_range] regardless of
        # sequence length (it's a weighted *average*, not a weighted *sum*), while still
        # preserving order-sensitivity: swapping two positions still changes the result
        # whenever their directions differ.
        total = 0
        weight_sum = 0
        for j, d in enumerate(directions):
            if j == index:
                continue
            weight = j + 1
            total += d * weight
            weight_sum += weight
        if weight_sum > 0:
            # Round back to an int: `position` (and the value this function returns)
            # is used directly as a deflector index downstream (`t % n` in ricochet()),
            # which requires an int -- the normalization above only needs to keep the
            # *magnitude* bounded, not switch the representation to floats.
            total = round(total / weight_sum)
        position += total
        return position

    def display_deflectors(self):
        for i, deflect in enumerate(self.deflectors_list):
            print(f"Deflector {i}: {deflect}")

    def clone(self):
      # Create a lightweight dummy instance
      new_brain = Ball_Trap_Brain(1, self.ball_energy // 2, self.alphabet, self.output,
                                   self.holes_multiplier, self.direction_range, self.max_bounces)

      # Fast shallow copy of the 10,000 elements
      new_brain.deflectors_list = [[d, v] for d, v in self.deflectors_list]
      new_brain.alphabet_directions = self.alphabet_directions[:]
      new_brain.alphabet_start_positions = self.alphabet_start_positions[:]

      return new_brain

    def shift_alphabet(self):
        random.shuffle(self.alphabet)
        for i in range(len(self.deflectors_list)):
          if self.deflectors_list[i][1] in self.output:
            idx = self.output.index(self.deflectors_list[i][1])
            self.deflectors_list[i][1] = self.alphabet[idx]

    def Mutate(self, individual_probability=0.1, probability=0.05):
        if random.random() < individual_probability:
            for i in range(len(self.deflectors_list)):
                  if random.random() < probability:
                      self.deflectors_list[i][0] = random.randint(0, len(self.deflectors_list)-1)

            for i in range(len(self.deflectors_list)):
                 if random.random() < probability:
                     self.deflectors_list[random.randint(0, len(self.deflectors_list)-1)][1] = random.choice(self.output)

            for i in range(len(self.alphabet_directions)):
                 if random.random() < probability:
                     self.alphabet_directions[i] = random.randint(-self.direction_range, self.direction_range)

            for i in range(len(self.alphabet_start_positions)):
                  if random.random() < probability:
                      self.alphabet_start_positions[i] = random.randint(0, len(self.deflectors_list)-1)

            if random.random() < probability:
                  self.set_holes(self.output, len(self.deflectors_list), self.holes_multiplier)

        return self

    def train_continuous(self, test_cases, epochs=300, steps=None, lr_start=0.05, lr_end=None, batch_size=16,
                          temp_start=2.5, temp_end=0.5, sharpness=0.75,
                          index_temp_start=None, index_temp_end=0.05, index_temp_power=2.0,
                          grad_clip_norm=1.0, restore_best=True,
                          repeat_penalty_weight=0.3, device=None,
                          warm_start=True, commit=True, verbose=True):
        """
        Continuous relaxation training.

        Every discrete choice the brain makes is replaced with a differentiable
        stand-in for the duration of training:
          - each deflector's destination index      -> a softmax over all deflectors
          - each deflector's hole/bounce identity    -> a sigmoid (hole probability)
          - each deflector's output symbol (if hole) -> a softmax over self.output
          - each deflector's bounce value (if not)   -> a free real-valued scalar
          - each alphabet direction (integer in [-direction_range, direction_range]) -> a
            softmax over that candidate range (same pattern as the start position below)
          - each alphabet start position              -> a softmax over all deflectors

        The ricochet loop is unrolled for `cap` steps and the hard indexing /
        branching / while-loop of the discrete simulator is replaced by
        probability-weighted ("expected value") versions of the same
        arithmetic, so the whole forward pass is differentiable and can be
        trained with gradient descent against the task loss instead of the
        genetic algorithm's mutate-and-select loop. A temperature is annealed
        from `temp_start` down to `temp_end` over training so the softmax
        distributions sharpen towards one-hot (i.e. towards the original
        discrete model) as training progresses.

        **Epochs vs. steps.** Each epoch is a full pass over a shuffled copy of
        `test_cases`, split into batches of `batch_size`; the optimizer takes one
        gradient step per batch (so an epoch is several steps, not one). Every
        example is seen exactly once per epoch, rather than each iteration
        drawing an independent random sample of the data (which gives no
        per-epoch coverage guarantee and can revisit or skip examples). The
        temperature schedule and the returned histories are still indexed by
        epoch (one entry per epoch, averaged across that epoch's batches/steps).

        Pass `steps=N` to train for exactly N gradient steps instead of a
        number of full passes -- useful when you want a training budget that
        doesn't depend on `len(test_cases)`, or want to compare runs at equal
        step counts across different dataset sizes. When `steps` is given, it
        takes over as the loop's unit: temp/index_temp/lr are annealed by
        `step/steps` instead of `epoch/epochs`, and `loss_history`/
        `accuracy_history` get one entry per step instead of one per epoch.
        Coverage is still epoch-like under the hood -- each step pulls the next
        `batch_size` examples off a shuffled copy of `test_cases`, reshuffling
        and starting a fresh pass whenever it runs out -- so examples are still
        seen roughly equally often, rather than each step drawing an
        independent random sample (which would revisit some examples far more
        than others over a long run). Leave `steps=None` (the default) to keep
        the original epoch-based behavior exactly as-is.

        Because the discrete model can emit a variable number of output
        symbols per input, this relaxation makes one simplifying assumption:
        it treats the *first hole reached* by each character's trajectory as
        that character's contribution to the output, and aligns those
        per-character predictions against the target string position by
        position, right-aligned (the last trajectory position lines up with
        the last target character, and so on).

        Every trajectory position is trained, not just the aligned ones.
        `never_fired = 1 - out_dist.sum()` (the probability a trajectory ends
        the simulation without ever emitting anything) turns each position's
        prediction into a genuine (K+1)-way distribution: the K real output
        symbols, plus one "emit nothing" class, all summing to 1 by
        construction. Leading input positions that fall outside the aligned
        window (i.e. that have no corresponding target character at all) are
        given "emit nothing" as their target class in that same distribution.
        Earlier versions of this method left those leading positions with no
        training signal whatsoever, so they were free to fire and leak
        arbitrary symbols into the real, full output regardless of how low
        the loss on the aligned positions got; a later attempt fixed that
        with a second, separately-weighted penalty term, but that fought the
        main loss for control of shared deflectors and could destabilize
        training. Folding "should this position emit at all" into the *same*
        per-position cross-entropy as the main task avoids both problems --
        there's one loss, and every position (aligned or not) has a
        well-defined target under it.

        There's a related, narrower pathology this doesn't cover: once a
        trajectory reaches a hole, this objective stops caring what happens
        for the rest of that trajectory's `cap` steps. In the real, discrete
        brain, if that trajectory keeps landing on holes afterward (a direct
        self-loop, a 2-cycle between two holes, or any longer cycle -- all
        equally invisible to the "first hole" objective above), `ricochet`
        just keeps re-appending symbols for the rest of the `cap` steps (e.g.
        `"3333333333"`, `"2323232323"`). `repeat_penalty_weight` (default 0.3)
        penalizes the expected number of *additional* hole-hits after the
        first one for every trajectory in the batch (`extra_fire` from
        `_c_simulate`) -- computed from the same per-example forward pass
        already used for the main loss, so it directly targets whatever
        cycle each specific trajectory is actually at risk of, rather than
        only guarding against direct self-loops.

        **`index_temp_start`/`index_temp_end`.** The per-step routing softmaxes
        (dest, symbol, hole, start-position logits) can always be sharpened by
        training regardless of `temp`, since their logits are free parameters that
        gradient descent can grow arbitrarily to compensate. But the very first
        step of each trajectory -- converting the continuous `travel_i` scalar into
        a distribution over starting deflectors, in `_c_soft_index_dist` -- has NO
        learnable parameters; its width is fixed purely by temperature/sharpness.
        If that temperature never anneals close to zero, this kernel stays diffuse
        for the entire run, so a meaningful fraction of probability mass leaks onto
        neighboring (wrong) starting deflectors no matter how well every other
        parameter is trained -- putting a hard floor under the loss and capping
        accuracy below 100%, however long training runs. `index_temp` is therefore
        annealed on its own, separate schedule from `temp`, defaulting to a much
        lower end value (`index_temp_end=0.05`) so this kernel actually reaches
        near one-hot by the end of training. `index_temp_start` defaults to
        `temp_start` if left as `None`.

        **Why loss could spike back up late in training (and the fixes below).**
        Every softmax's gradient w.r.t. its logits scales roughly like `1/temp`, so
        as `temp`/`index_temp` cool, the same size of parameter update produces
        increasingly large swings in output probabilities -- and the (non-learnable)
        `_c_soft_index_dist` kernel simultaneously goes from forgiving (wide, so small
        errors in the continuous `travel_i` estimate get gently split across
        neighboring deflectors) to unforgiving (narrow, so the same small error can
        dump all its mass onto the wrong deflector). With a flat learning rate and no
        clipping, this means training becomes progressively *less* stable right when
        it's cooling into its most precision-demanding phase, and a single overshoot
        can flip a class assignment and spike the loss. Four things address this:

        - `lr_start`/`lr_end`: the learning rate now decays (cosine) alongside
          `temp`/`index_temp` instead of staying flat, so the effective step size in
          probability-space doesn't grow as the softmaxes sharpen. Defaults to
          `lr_end = lr_start / 20`.
        - `grad_clip_norm`: gradients are clipped to this global norm (via
          `torch.nn.utils.clip_grad_norm_`) before every `optimizer.step()`, capping
          how large a single overshoot can be regardless of temperature. Set to
          `None` to disable.
        - `index_temp_power`: the index-kernel cooling schedule is no longer linear
          in epoch -- it's `index_temp_start + (index_temp_end - index_temp_start) *
          frac ** index_temp_power`. With the default `power=2.0` this spends more
          epochs in the middle-to-low range before dropping to its sharpest, giving
          `travel_i` more time to converge before precision demands peak, rather than
          tightening the kernel at a constant rate all the way to the end.
        - `restore_best`: the parameter state with the lowest epoch-average
          `task_loss` seen during training is checkpointed (in memory) and restored
          onto the instance at the end, so a late-training spike can't leave you with
          a worse model than one that existed earlier in the run. Set to `False` to
          just keep whatever the final epoch produced (the old behavior).

        The trained continuous parameters are kept on this instance (usable
        via `predict_continuous` / `evaluate_continuous`, see below) whether
        or not you discretize. If `commit=True` (the default), every
        continuous parameter is *also* discretized (argmax / sign / rounding)
        and written directly back into this instance's own `deflectors_list`,
        `alphabet_directions` and `alphabet_start_positions` attributes, so
        every other method on this class (compute_input, ricochet, clone,
        Mutate, display_deflectors, ...) keeps working exactly as before,
        just with gradient-trained values in place of the old ones. Pass
        `commit=False` to leave those discrete attributes untouched and only
        keep the trained continuous version (e.g. if you want to compare
        continuous vs. discrete behavior, or keep iterating with
        `train_continuous` again before ever committing to a discrete brain).

        Returns `(loss_history, accuracy_history)`, both per-epoch lists of
        floats (one entry per full pass over the data, averaged across that
        epoch's batches) -- these always reflect what actually happened each
        epoch, even if `restore_best=True` rewound the instance's live
        parameters to an earlier, better checkpoint afterward. `accuracy_history` is a cheap "token accuracy" over
        *every* trajectory position seen during the epoch (aligned or blank):
        it checks whether that position's predicted class -- its top symbol if
        its emit-probability is >= 0.5, else "blank" -- matches the target
        class, and reports the fraction that do. It's computed for free from
        the same numbers already used for the loss -- no extra simulation is
        run -- so while it now also reflects whether leading positions are
        correctly suppressed, it's still not the same as full-sequence
        exact-match correctness on the real, discrete brain (see
        `evaluate_continuous` / `evaluate_performance` for that).
        """
        import torch

        self._init_continuous_params(warm_start=warm_start, sharpness=sharpness, device=device)
        c = self._c_meta
        output_index = c['output_index']

        params = [self._c_dest_logits, self._c_hole_logit, self._c_symbol_logits,
                  self._c_bounce_val, self._c_direction_logits, self._c_start_logits]
        optimizer = torch.optim.Adam(params, lr=lr_start)

        if index_temp_start is None:
            index_temp_start = temp_start
        if lr_end is None:
            lr_end = lr_start / 20.0

        best_loss = float('inf')
        best_state = None  # list of detached, cloned tensors matching `params`, if restore_best

        history = []
        acc_history = []
        n_batches = max(1, (len(test_cases) + batch_size - 1) // batch_size)
        temp = temp_start
        index_temp = index_temp_start
        import math

        use_steps = steps is not None
        total_units = steps if use_steps else epochs
        unit_label = 'step' if use_steps else 'epoch'

        # Only used when use_steps=True: a shuffled copy of the data that we
        # slice through `batch_size` at a time, reshuffling and starting over
        # whenever it runs out. This gives each step's batch the same
        # "everyone gets seen equally often" coverage as the epoch loop below,
        # instead of each step drawing an independent random sample (which
        # would let some examples go unseen for arbitrarily long stretches).
        step_pool = []
        step_cursor = 0

        for unit in range(total_units):
            frac = unit / max(1, total_units - 1)
            temp = temp_start + (temp_end - temp_start) * frac
            # Anneal the (non-learnable) index kernel on its own schedule so it
            # actually reaches near one-hot by the end of training instead of
            # staying capped wherever `temp` (used for the learnable logit
            # softmaxes) happens to end up. Raised to `index_temp_power` (default
            # 2.0) rather than linear: spends more units in the middle-to-low
            # range so `travel_i` has time to converge before the kernel gets to
            # its sharpest, least-forgiving point, instead of tightening at a
            # constant rate all the way to the end.
            index_temp = index_temp_start + (index_temp_end - index_temp_start) * (frac ** index_temp_power)
            # Cosine-decay the learning rate alongside temp/index_temp so the
            # effective step size in probability-space doesn't grow as the
            # softmaxes sharpen (softmax gradients scale like 1/temp).
            lr_now = lr_end + 0.5 * (lr_start - lr_end) * (1 + math.cos(math.pi * frac))
            for group in optimizer.param_groups:
                group['lr'] = lr_now

            if use_steps:
                if step_cursor + batch_size > len(step_pool):
                    step_pool = list(test_cases)
                    random.shuffle(step_pool)
                    step_cursor = 0
                batches = [step_pool[step_cursor:step_cursor + batch_size]]
                step_cursor += batch_size
            else:
                shuffled = list(test_cases)
                random.shuffle(shuffled)
                batches = [shuffled[b * batch_size:(b + 1) * batch_size] for b in range(n_batches)]

            epoch_loss_sum = 0.0
            epoch_count = 0
            epoch_correct = 0

            for batch in batches:
                if not batch:
                    continue

                optimizer.zero_grad()
                # Computed once per batch (not per example, and not once per epoch --
                # reusing the same autograd graph across multiple backward() calls
                # within an epoch would error on the second batch) and reused for
                # every example in this batch below, instead of _c_simulate
                # redundantly rebuilding the same O(D^2) dest_probs softmax (etc.)
                # from scratch for every single example. Larger batch_size means
                # this saving compounds across more examples per recompute.
                precomputed = self._c_precompute(temp)
                _dev = precomputed['dest_probs'].device
                total_loss = torch.tensor(0.0, device=_dev)
                count = 0
                correct_t = torch.zeros((), dtype=torch.float32, device=_dev)  # accumulate as a tensor; one .item() per batch, not per position
                repeat_loss = torch.tensor(0.0, device=_dev)
                repeat_count = 0
                for inp, target in batch:
                    tokens = TokenizeInput(inp, self.alphabet)
                    indices = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
                    if not indices:
                        continue
                    outs, never_fired, extra_fire = self._c_simulate(indices, temp, index_temp, precomputed=precomputed)
                    target_indices = [output_index[ch] for ch in target if ch in output_index]
                    L = len(outs)
                    n = min(L, len(target_indices))
                    # Right-align: the *last* n trajectory positions correspond to the *last* n
                    # target characters. Test inputs typically share a literal, invariant prefix
                    # (e.g. every "1+i" starts with the same "1+"), so left-aligning would
                    # repeatedly score that shared, input-invariant trajectory against different
                    # targets -- gradients then can't distinguish examples at all. The trailing
                    # characters of the input are where the example-specific information (e.g.
                    # the digits of i) actually lives, so align from the end instead.
                    out_offset = L - n
                    tgt_offset = len(target_indices) - n
                    # Score EVERY position, not just the aligned tail. `out_dist[k]` and
                    # `never_fired` already form one consistent (K+1)-way distribution over
                    # "emit symbol k" / "emit nothing" (never_fired = 1 - out_dist.sum() by
                    # construction), so positions before out_offset -- which have no target
                    # character at all -- are simply given "emit nothing" as their target class
                    # in that SAME distribution, rather than being left with no training signal
                    # (which is what let them leak arbitrary symbols into the real output) or
                    # fought over with a separately-weighted penalty term (which destabilized
                    # training by competing against the main loss on shared deflectors).
                    #
                    # This whole block is vectorized across the L positions of a single example
                    # instead of a Python loop that called .item() on every position (each of
                    # which forces a GPU/CPU sync) -- building the small, cheap target_class list
                    # in Python is fine, but the actual tensor math (gather/where/argmax) below
                    # runs once per example instead of once per character.
                    target_class_list = [c['K']] * out_offset + [
                        target_indices[tgt_offset + (i - out_offset)] for i in range(out_offset, L)
                    ]
                    outs_stack = torch.stack(outs, dim=0)                     # (L, K)
                    never_fired_stack = torch.stack(never_fired)              # (L,)
                    target_class_t = torch.tensor(target_class_list, device=outs_stack.device)  # (L,)
                    is_blank = target_class_t == c['K']
                    safe_idx = target_class_t.clamp(max=c['K'] - 1)
                    gathered = outs_stack.gather(1, safe_idx.unsqueeze(1)).squeeze(1)  # (L,)
                    target_prob = torch.where(is_blank, never_fired_stack, gathered)
                    total_loss = total_loss + (-torch.log(target_prob + 1e-8)).sum()

                    p_emit = outs_stack.sum(dim=1)                             # (L,)
                    pred_symbol = outs_stack.argmax(dim=1)                     # (L,)
                    pred_class = torch.where(p_emit >= 0.5, pred_symbol,
                                              torch.full_like(pred_symbol, c['K']))
                    correct_t = correct_t + (pred_class == target_class_t).sum()
                    count += L

                    repeat_loss = repeat_loss + torch.stack(extra_fire).sum()
                    repeat_count += L

                if count == 0:
                    continue
                task_loss = total_loss / count
                combined_loss = task_loss
                if repeat_penalty_weight and repeat_count:
                    combined_loss = combined_loss + repeat_penalty_weight * (repeat_loss / repeat_count)
                combined_loss.backward()
                if grad_clip_norm is not None:
                    torch.nn.utils.clip_grad_norm_(params, grad_clip_norm)
                optimizer.step()

                epoch_loss_sum += task_loss.item() * count
                epoch_count += count
                epoch_correct += int(correct_t.item())

            if epoch_count == 0:
                continue
            avg_loss = epoch_loss_sum / epoch_count
            avg_acc = epoch_correct / epoch_count
            history.append(avg_loss)
            acc_history.append(avg_acc)

            if restore_best and avg_loss < best_loss:
                best_loss = avg_loss
                best_state = [p.detach().clone() for p in params]

            if verbose and (unit % max(1, total_units // 20) == 0 or unit == total_units - 1):
                print(f"[continuous] {unit_label} {unit}: loss={avg_loss:.4f} token_acc={avg_acc:.3f} "
                      f"temp={temp:.2f} index_temp={index_temp:.3f} lr={lr_now:.5f} "
                      f"batches={len(batches)}")

        # Remember the temperatures training ended at, so predict_continuous /
        # evaluate_continuous default to something close to what was actually trained.
        self._c_meta['last_temp'] = temp
        self._c_meta['last_index_temp'] = index_temp

        if restore_best and best_state is not None:
            with torch.no_grad():
                for p, saved in zip(params, best_state):
                    p.copy_(saved)
            if verbose:
                print(f"[continuous] restored best checkpoint: loss={best_loss:.4f} "
                      f"(final epoch's loss was {history[-1]:.4f})")

        if commit:
            self._discretize_continuous_into(self)

        return history, acc_history

    def _init_continuous_params(self, warm_start=True, sharpness=0.75, device=None):
        """Build (or rebuild) this instance's continuous relaxation parameters from its
        current discrete state, storing them on the instance so they persist beyond a
        single train_continuous call and can be used by predict_continuous /
        evaluate_continuous / _discretize_continuous_into.

        `device` defaults to CUDA if available, else CPU. All the heavy per-step
        tensor ops in _c_simulate (softmaxes, matmuls) run wherever these
        parameters live, so training on a GPU (when available) can be
        substantially faster than the CPU default, especially with larger
        `batch_size`/`deflectors` -- pass `device='cpu'` explicitly to force CPU.
        """
        import torch

        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'

        D = len(self.deflectors_list)
        A = len(self.alphabet)
        K = len(self.output)
        output_index = {sym: k for k, sym in enumerate(self.output)}
        positions = torch.arange(D, dtype=torch.float32, device=device)
        abs_diff = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()  # abs_diff[d, d'] = |d - d'|

        V = self.direction_range
        n_dirs = 2 * V + 1
        direction_values = torch.arange(-V, V + 1, dtype=torch.float32, device=device)  # (n_dirs,) candidate integers

        CONF = 4.0  # logit magnitude used so a warm-started softmax closely matches the current discrete choice
        dest_logits = torch.zeros(D, D, device=device)
        hole_logit = torch.zeros(D, device=device)
        symbol_logits = torch.zeros(D, K, device=device)
        bounce_val = torch.zeros(D, device=device)
        direction_logits = torch.zeros(A, n_dirs, device=device)
        start_logits = torch.zeros(A, D, device=device)

        if warm_start:
            for i, (dest, val) in enumerate(self.deflectors_list):
                dest_logits[i, dest] = CONF
                if isinstance(val, int):
                    hole_logit[i] = -CONF
                    bounce_val[i] = float(val)
                else:
                    hole_logit[i] = CONF
                    symbol_logits[i, output_index[val]] = CONF
            for i, d in enumerate(self.alphabet_directions):
                # Clamp in case direction_range was narrowed since `d` was chosen/mutated.
                idx = max(0, min(n_dirs - 1, int(d) + V))
                direction_logits[i, idx] = CONF
            for i, p in enumerate(self.alphabet_start_positions):
                start_logits[i, p] = CONF
        else:
            dest_logits.normal_(0, 0.1)
            hole_logit.normal_(0, 0.1)
            symbol_logits.normal_(0, 0.1)
            bounce_val.normal_(0, 1.0)
            direction_logits.normal_(0, 0.1)
            start_logits.normal_(0, 0.1)

        for p in (dest_logits, hole_logit, symbol_logits, bounce_val, direction_logits, start_logits):
            p.requires_grad_(True)

        self._c_dest_logits = dest_logits
        self._c_hole_logit = hole_logit
        self._c_symbol_logits = symbol_logits
        self._c_bounce_val = bounce_val
        self._c_direction_logits = direction_logits
        self._c_start_logits = start_logits
        self._c_meta = dict(D=D, A=A, K=K, cap=self.max_bounces, ball_energy=float(self.ball_energy),
                             output_index=output_index, positions=positions, abs_diff=abs_diff,
                             direction_values=direction_values,
                             sharpness=sharpness, last_temp=None, last_index_temp=None)

    def _c_soft_index_dist(self, travel_values, temp):
        """Batched: travel_values is a 1D tensor of shape (L,) (one travel value per
        input position); returns a (L, D) tensor, one starting-deflector
        distribution per row."""
        import torch
        import torch.nn.functional as F
        c = self._c_meta
        diff = c['positions'].unsqueeze(0) - travel_values.unsqueeze(1)  # (L, D)
        diff = torch.remainder(diff + c['D'] / 2, c['D']) - c['D'] / 2
        logits = -(diff ** 2) / (c['sharpness'] * temp)
        return F.softmax(logits, dim=1)

    def _c_precompute(self, temp):
        """Compute the tensors that _c_simulate needs but that only depend on the
        current parameter values and `temp` -- NOT on which example is being
        simulated: dest_probs, hole_p, symbol_probs, weighted_abs, and the
        expected per-token direction/start-position. `temp` is constant across an
        entire training epoch (it's only updated once per epoch, not per batch or
        per example), so previously calling _c_simulate separately for every
        example recomputed this same O(D^2) softmax (among others) from scratch
        once per example, all producing an identical result -- pure wasted work
        that scaled with batch_size * n_batches for no benefit. Call this once per
        epoch (or whenever `temp` changes) and pass the result into every
        _c_simulate call via its `precomputed` argument instead."""
        import torch
        import torch.nn.functional as F

        c = self._c_meta
        dest_logits, hole_logit = self._c_dest_logits, self._c_hole_logit
        symbol_logits, bounce_val = self._c_symbol_logits, self._c_bounce_val
        direction_logits, start_logits = self._c_direction_logits, self._c_start_logits

        direction_probs_all = F.softmax(direction_logits / temp, dim=1)             # (A, n_dirs)
        expected_direction_all = (direction_probs_all * c['direction_values']).sum(dim=1)  # (A,)

        start_probs_all = F.softmax(start_logits / temp, dim=1)    # (A, D)
        expected_start_all = (start_probs_all * c['positions']).sum(dim=1)  # (A,)

        dest_probs = F.softmax(dest_logits / temp, dim=1)          # (D, D)
        hole_p = torch.sigmoid(hole_logit)                         # (D,)
        symbol_probs = F.softmax(symbol_logits / temp, dim=1)      # (D, K)
        weighted_abs = (dest_probs * c['abs_diff']).sum(dim=1)     # (D,) expected hop distance

        return dict(expected_direction_all=expected_direction_all, expected_start_all=expected_start_all,
                    dest_probs=dest_probs, hole_p=hole_p, symbol_probs=symbol_probs,
                    weighted_abs=weighted_abs, bounce_val=bounce_val)

    def _c_simulate(self, input_indices, temp, index_temp=None, precomputed=None):
        """Run the differentiable, soft ricochet simulation using this instance's
        persisted continuous parameters (self._c_*). Returns (outputs, never_fired,
        extra_fire), one entry per input character: `outputs[i]` is a probability
        distribution over self.output (via the "first hole reached" convention -- see
        train_continuous's docstring), `never_fired[i]` is the probability that
        trajectory never emits anything within `cap` steps, and `extra_fire[i]` is the
        expected number of *additional* hole-hits after the first one (i.e. how much
        the trajectory keeps re-firing instead of settling down for the rest of its
        cap steps).

        `temp` controls every *logit-based* softmax (dest_probs, symbol_probs,
        start_probs_all): those can always be sharpened by training regardless of
        temp, since the logits themselves are free parameters that gradient descent
        can grow without bound to compensate for a high temp.

        `index_temp` controls only `_c_soft_index_dist`, the kernel that turns the
        continuous `travel_i` scalar into a distribution over starting deflectors.
        That kernel has NO learnable parameters -- its width is fixed entirely by
        temp/sharpness -- so unlike the logit-based softmaxes above, nothing during
        training can sharpen it beyond whatever index_temp dictates. If index_temp
        stays high, the starting-state distribution stays diffuse forever, capping
        the maximum achievable target_prob (and hence putting a floor under the
        loss) no matter how well the other parameters train. `index_temp` therefore
        needs to anneal much lower than `temp` by the end of training. Defaults to
        `temp` for backward compatibility, but train_continuous now passes its own,
        more aggressively annealed schedule.

        `precomputed` is an optional dict from `_c_precompute(temp)` (dest_probs,
        hole_p, symbol_probs, weighted_abs, bounce_val, and the expected per-token
        direction/start-position) -- pass it in when calling this many times at the
        same `temp` (e.g. once per example within a training epoch) to skip
        redundantly recomputing those from raw logits on every call. Left as
        `None` (computed fresh internally) for one-off calls like a single
        predict_continuous.

        The per-character simulation is fully vectorized across all L input
        positions at once (an (L, D) state tensor advances through `cap` batched
        matmuls) rather than looping over characters in Python -- mathematically
        identical to running each character's trajectory separately, just
        computed in parallel.
        """
        import torch

        if index_temp is None:
            index_temp = temp

        c = self._c_meta
        if precomputed is None:
            precomputed = self._c_precompute(temp)
        expected_direction_all = precomputed['expected_direction_all']
        expected_start_all = precomputed['expected_start_all']
        dest_probs = precomputed['dest_probs']
        hole_p = precomputed['hole_p']
        symbol_probs = precomputed['symbol_probs']
        weighted_abs = precomputed['weighted_abs']
        bounce_val = precomputed['bounce_val']

        L = len(input_indices)
        dirs = expected_direction_all[input_indices]                # (L,)
        # Linear position weighting -- mirrors the discrete Brain.travel(): weight
        # token j's direction by (j+1) instead of a flat, order-invariant sum, so
        # swapping two positions changes travel_i (unless their directions happen
        # to match) instead of colliding on the same value the way every anagram
        # of the same characters used to.
        #
        # Normalized by the (leave-one-out) weight sum, exactly like the discrete
        # Brain.travel() -- without this, travel_i's magnitude grows with L (up to
        # ~quadratically), which for long sequences pushes travel_i many multiples of
        # D past the soft-index kernel's useful range, effectively randomizing the
        # starting-deflector distribution regardless of token content and collapsing
        # the output down to just a few trajectories that happen to reach a hole.
        pos_weights = torch.arange(1, L + 1, dtype=dirs.dtype, device=dirs.device)  # (L,)
        weighted_dirs = dirs * pos_weights                         # (L,)
        total_weighted = weighted_dirs.sum()
        starts = expected_start_all[input_indices]                 # (L,)

        # Leave-one-out weight sum per position: total weight (1 + 2 + ... + L) minus
        # this position's own weight, matching the leave-one-out numerator above.
        total_weight = pos_weights.sum()
        weight_sum_loo = (total_weight - pos_weights).clamp(min=1e-8)  # (L,) avoid /0 when L==1

        # All L positions' trajectories advance together as one (L, D) batch --
        # each row is independent (matches running the per-character loop below,
        # just done for every character in parallel instead of one at a time).
        travel_all = starts + (total_weighted - weighted_dirs) / weight_sum_loo  # (L,) matches Brain.travel()
        state = self._c_soft_index_dist(travel_all, index_temp)    # (L, D)

        be = torch.full((L,), c['ball_energy'], dtype=state.dtype, device=state.device)
        alive = torch.ones(L, dtype=state.dtype, device=state.device)         # soft "be > 0 and cap_idx < cap" gate
        not_hole_yet = torch.ones(L, dtype=state.dtype, device=state.device)  # soft "no hole reached yet" gate
        out_dist = torch.zeros(L, c['K'], dtype=state.dtype, device=state.device)
        extra = torch.zeros(L, dtype=state.dtype, device=state.device)        # expected re-fires after the first hole

        for _ in range(c['cap']):
            p_hole = (state * hole_p).sum(dim=1)                             # (L,)
            p_bounce_energy = (state * (1 - hole_p) * bounce_val).sum(dim=1)  # (L,)
            exp_dist = (state * weighted_abs).sum(dim=1)                      # (L,)

            already_fired = alive * (1.0 - not_hole_yet)
            extra = extra + already_fired * p_hole

            p_first_hole = alive * not_hole_yet * p_hole                      # (L,)
            sym_dist = (state * hole_p.unsqueeze(0)) @ symbol_probs           # (L, K)
            sym_dist = sym_dist / (p_hole.unsqueeze(1) + 1e-8)
            out_dist = out_dist + p_first_hole.unsqueeze(1) * sym_dist

            energy_delta = p_hole + p_bounce_energy - exp_dist
            be = be + energy_delta
            alive = alive * torch.sigmoid(be * 2.0)
            not_hole_yet = not_hole_yet * (1 - p_hole)

            state = state @ dest_probs                                       # (L, D) @ (D, D) -> (L, D)

        outputs = list(out_dist.unbind(dim=0))                    # L tensors of shape (K,), same API as before
        never_fired = list((1.0 - out_dist.sum(dim=1)).unbind(dim=0))
        extra_fire = list(extra.unbind(dim=0))

        return outputs, never_fired, extra_fire


    def _discretize_continuous_into(self, target_brain):
        """Argmax/sign/round this instance's continuous parameters (self._c_*) and write
        them into target_brain's discrete attributes (deflectors_list, alphabet_directions,
        alphabet_start_positions)."""
        import torch
        c = self._c_meta
        target_brain.direction_range = self.direction_range
        with torch.no_grad():
            for i in range(c['D']):
                dest = int(torch.argmax(self._c_dest_logits[i]).item())
                if torch.sigmoid(self._c_hole_logit[i]).item() > 0.5:
                    val = self.output[int(torch.argmax(self._c_symbol_logits[i]).item())]
                else:
                    val = int(round(self._c_bounce_val[i].item()))
                target_brain.deflectors_list[i][0] = dest
                target_brain.deflectors_list[i][1] = val
            for i in range(c['A']):
                dir_idx = int(torch.argmax(self._c_direction_logits[i]).item())
                target_brain.alphabet_directions[i] = int(c['direction_values'][dir_idx].item())
                target_brain.alphabet_start_positions[i] = int(torch.argmax(self._c_start_logits[i]).item())

    def commit_continuous(self):
        """Discretize this instance's already-trained continuous parameters (from a prior
        train_continuous(..., commit=False) call) into its discrete deflectors_list /
        alphabet_directions / alphabet_start_positions, without training any further."""
        if not hasattr(self, '_c_meta'):
            raise RuntimeError("No continuous parameters to commit -- call train_continuous(...) first.")
        self._discretize_continuous_into(self)

    def predict_continuous(self, input_str, temp=None, index_temp=None, return_probs=False):
        """
        Answer `input_str` using the *continuous* (un-discretized) brain directly --
        no argmax/rounding of parameters, no dependency on deflectors_list/
        alphabet_directions/alphabet_start_positions at all. Requires a prior
        train_continuous(...) call (with or without commit=True) on this instance.

        Unlike training (which only scores a target-aligned tail of each trajectory),
        this decodes *every* input character's trajectory: if its "never fire" probability
        is below 0.5 it's treated as emitting, and the most likely symbol from its output
        distribution is appended, in the same left-to-right order compute_input uses.

        `temp` defaults to the (logit-softmax) temperature training ended at, and
        `index_temp` defaults to the separately-annealed index-kernel temperature
        training ended at (see train_continuous's docstring for why these are two
        different schedules) -- colder is closer to the discretized/deterministic
        behavior; pass higher values to see softer, more exploratory behavior. Set
        `return_probs=True` to also get back, for every input character,
        `(probability it emits at all, {symbol: probability})`.
        """
        import torch

        if not hasattr(self, '_c_meta'):
            raise RuntimeError(
                "No continuous parameters on this brain yet -- call train_continuous(...) "
                "first (pass commit=False if you don't want it to also overwrite the "
                "discrete deflectors_list/alphabet_directions/alphabet_start_positions)."
            )
        c = self._c_meta
        if temp is None:
            temp = c['last_temp'] if c['last_temp'] is not None else 0.5
        if index_temp is None:
            index_temp = c['last_index_temp'] if c['last_index_temp'] is not None else 0.05

        tokens = TokenizeInput(input_str, self.alphabet)
        indices = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
        if not indices:
            return ("", []) if return_probs else ""

        with torch.no_grad():
            outs, never_fired, _extra_fire = self._c_simulate(indices, temp, index_temp)

        decoded = []
        probs = []
        for out_dist, nf in zip(outs, never_fired):
            p_emit = 1.0 - nf.item()
            sym_probs = {sym: p for sym, p in zip(self.output, out_dist.tolist())}
            probs.append((p_emit, sym_probs))
            if p_emit >= 0.5:
                decoded.append(self.output[int(torch.argmax(out_dist).item())])

        result = "".join(decoded)
        return (result, probs) if return_probs else result

    def evaluate_continuous(self, test_cases, temp=None, index_temp=None, ratio=False):
        """Like evaluate_performance, but scores predict_continuous (the un-discretized
        brain) against test_cases instead of compute_input (the discretized/discrete
        brain). Returns the average score (0/1 exact-match per case, or a 0-1 similarity
        ratio if ratio=True)."""
        total = 0.0
        for inp, answer in test_cases:
            pred = self.predict_continuous(inp, temp=temp, index_temp=index_temp)
            if ratio:
                total += difflib.SequenceMatcher(None, pred, answer).ratio()
            else:
                total += 1 if pred == answer else 0
        return total / len(test_cases)

## Test case generation

In [ ]:
import random
import string
def generate_test_cases(amount, number, indiv=True):

  test_cases = []
  if indiv == True:
    test_cases = [(f'1+{i}', f"{random.randint(0, 1000)}") for i in range(amount)]
  else:
    x = list("this module contains functions to evaluate the performance of individual brains and entire populations against test cases using string similarity metrics")
    for i in range(len(x)):
      a = ''
      if i != len(x)-1:
        b = x[i+1]
      for j in range(i+1):
        if i-j <= amount:
          a += x[j]
      test_cases = [ ("cosine-decays alongside ", " temp/index_temp for")]


  return test_cases

## Fitness evaluation

Fitness evaluation module for Ball_Trap_Brain genetic algorithm.

This module contains functions to evaluate the performance of individual brains
and entire populations against test cases using string similarity metrics.


In [ ]:
import difflib
import math

In [ ]:
def evaluate_performance(input_val, answer, brain, ratio=False):

    output = brain.compute_input(input_val)
    if ratio:
        return difflib.SequenceMatcher(None, ''.join(output), answer).ratio()
    else:
      return 1 if ''.join(output) == answer else 0

In [ ]:
import concurrent.futures

In [ ]:
def eval_single_brain(args):
    individual, test_cases, ratio = args
    avg_fitness = sum(
        evaluate_performance(inp, ans, individual, ratio) for inp, ans in test_cases
    ) / len(test_cases)
    return avg_fitness

In [ ]:
def evaluate_population_parallel(population, test_cases, ratio=False):
    # Package arguments for the worker pool
    args = [(ind, test_cases, ratio) for ind in population]

    # Run across available CPU cores
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(executor.map(eval_single_brain, args))

    best_fitness = max(results)
    best_brain = population[results.index(best_fitness)]
    return best_fitness, best_brain

## Genetic algorithm (`main`)

Genetic algorithm to evolve Ball_Trap_Brain instances to produce desired outputs.

This module runs a genetic algorithm that evolves a population of brains over
multiple generations. Each brain is evaluated against test cases, and the best
performer is mutated to create the next generation.


In [ ]:
string = "This module runs a genetic algorithm that evolves a population of brains over multiple generations. Each brain is evaluated against test cases, and the best performer is mutated to create the next generation."

In [ ]:
import copy
import difflib
import random
import shelve

Test cases: pairs of (input_string, expected_output_string) used for fitness evaluation

In [ ]:
alphabet = list("abcdefghijklmnopqrstuvwxyz ")  # Define the alphabet used in the brains
output = list("abcdefghijklmnopqrstuvwxyz ")

In [ ]:
def main(mini_batch, alphabet, generations=500, pop_size=100, brain=Ball_Trap_Brain(1000, 4, alphabet, output, 2), ratio=False):
    """
    Run the genetic algorithm for evolving Ball_Trap_Brain instances.

    Returns:
        Ball_Trap_Brain: The best brain found after evolution.
    """
    # Evolution hyperparameters
    population_size = pop_size

    # Initialize population with random brains
    # Parameters: 10 deflectors, 2 bounce energy, alphabet of 6 characters

    population = [brain.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
    population.append(brain)
    best_individual = None
    mini_batch_0 = copy.copy(mini_batch)
    #a = [mini_batch_0.pop(0) for i in range(10)]
    highest_score = -1
    # Run evolution loop
    for generation in range(generations):

        # Evaluate all brains and find the best performer
        best_fitness, best_individual = evaluate_population_parallel(population,mini_batch, ratio)



        # Check for perfect solution
        if best_fitness >= 1:
          print(f"Generation {generation}: Solution found!")
          return best_individual
          #for i in range(10):
            #a.append(mini_batch_0.pop(0))


        # Create next generation by mutating the best brain multiple times
        # New code: passes best_fitness, individual_probability, base_probability
        new_population = [best_individual.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
        population = new_population
        population.append(best_individual)
        print(f"Generation {generation}: Best fitness = {best_fitness}")
    return best_individual

In [ ]:
training_1 = generate_test_cases(amount=100, number='1', indiv=True)

## `Council_Brain` & council training

In [ ]:
from collections import Counter

In [ ]:
class Council_Brain:
    def __init__(self, brains):
        """
        Initializes the council with a list of separately trained brains.
        """
        self.brains = brains

    def compute_input(self, input_val):
        """
        Each brain processes the input separately.
        The council votes on the final answer by finding the most common output.
        """
        votes = []
        for brain in self.brains:
            # Each brain outputs a list of characters; join them to form the string
            output_str = brain.predict_continuous(input_val)
            votes.append(output_str)

        # Handle empty cases securely
        if not votes:
            return []

        # Tally the votes and find the majority
        vote_counts = Counter(votes)
        majority_vote = vote_counts.most_common(1)[0][0]

        # Return as a list of characters to perfectly match the original Ball_Trap_Brain format
        return list(majority_vote)

In [ ]:
def train_council(num_brains, training_data, alphabet, output_format, generations=50, pop_size=100):
    """
    Trains a council where each brain is trained separately using the genetic algorithm.
    """
    trained_brains = []

    for i in range(num_brains):
        print(f"\n=====================================")
        print(f" Training Council Brain {i+1} / {num_brains}")
        print(f"=====================================")
        base_brain = Ball_Trap_Brain(50, 10, alphabet, output, 2)
        # Initialize a fresh brain for this specific council member
        #base_brain = Ball_Trap_Brain(100, 10, alphabet, output_format, 2)

        # Train separately using the existing main GA loop
        for j in range(1):
          print(f'---{j}---')
          base_brain.train_continuous(
              random.sample(training_data,10),
              epochs=20,
              steps=generations,
              lr_start=0.4,
              lr_end=0.1,      # cosine-decays alongside temp/index_temp for late-training stability
              grad_clip_norm=1.0, # caps how large a single overshoot can be, regardless of temperature
              restore_best=False,  # roll back to the best checkpoint seen, in case of a late loss spike
              batch_size=1,
              warm_start=True,
              commit=False,   # keep the discrete brain untouched; only train the continuous version
              verbose=True,
              temp_start=2.5,
              temp_end=0.5,       # anneals the learnable-logit softmaxes (dest/symbol/hole/start)
              index_temp_start=2.5,
              index_temp_end=0.5,   # anneals the non-learnable start-index kernel much further down,
                                      # since nothing else can sharpen it -- see train_continuous's docstring
              index_temp_power=2.0,  # spends more epochs in the middle-to-low range before tightening
                                      # to its sharpest, least-forgiving point, instead of a flat linear cool
          )
        trained_brains.append(base_brain)

    return Council_Brain(trained_brains)

## Continuous relaxation training — demo

Trains a brain with gradient descent (requires `torch`; installed via `pip install torch` if missing) with `commit=False`, so the discrete brain is left completely untouched, and answers questions directly with the continuous version via `predict_continuous`. Then calls `commit_continuous()` and compares against the real, discrete `compute_input` -- with both pollution bugs fixed, these should now largely agree with each other, with no more repeated-character garbage.


In [ ]:
#!pip install torch --quiet   # uncomment if torch isn't already installed

In [ ]:
import random
#random.seed(0)

In [ ]:
demo_brain = Ball_Trap_Brain(100, 10, alphabet, output, 5, max_bounces=20)
cases = generate_test_cases(amount=1, number='1', indiv=False)  # '1+i' -> str(1+i)
random.shuffle(cases)
share = 1
demo_cases = cases[:int(share*len(cases))]
demo_test = cases[int(share*len(cases)):]
loss_history, acc_history = demo_brain.train_continuous(
    demo_cases,
    epochs=10,
    steps=1000,
    lr_start=0.8,
    lr_end=0.8,      # cosine-decays alongside temp/index_temp for late-training stability
    grad_clip_norm=1, # caps how large a single overshoot can be, regardless of temperature
    restore_best=False,  # roll back to the best checkpoint seen, in case of a late loss spike
    batch_size=1,
    warm_start=True,
    commit=False,   # keep the discrete brain untouched; only train the continuous version
    verbose=True,
    temp_start=2.5,
    temp_end=0.5,       # anneals the learnable-logit softmaxes (dest/symbol/hole/start)
    index_temp_start=2.5,
    index_temp_end=0.5,   # anneals the non-learnable start-index kernel much further down,
                       # since nothing else can sharpen it -- see train_continuous's docstring
    index_temp_power=2.0,  # spends more epochs in the middle-to-low range before tightening
                            # to its sharpest, least-forgiving point, instead of a flat linear cool
)
print('token_acc[0] ->', acc_history[0], ' token_acc[-1] ->', acc_history[-1])

[continuous] step 0: loss=3.2310 token_acc=0.043 temp=2.50 index_temp=2.500 lr=0.80000 batches=1
[continuous] step 50: loss=0.8370 token_acc=0.609 temp=2.40 index_temp=2.495 lr=0.80000 batches=1
[continuous] step 100: loss=0.8206 token_acc=0.696 temp=2.30 index_temp=2.480 lr=0.80000 batches=1
[continuous] step 150: loss=0.8141 token_acc=0.783 temp=2.20 index_temp=2.455 lr=0.80000 batches=1
[continuous] step 200: loss=0.8061 token_acc=0.826 temp=2.10 index_temp=2.420 lr=0.80000 batches=1
[continuous] step 250: loss=0.7944 token_acc=0.783 temp=2.00 index_temp=2.375 lr=0.80000 batches=1
[continuous] step 300: loss=0.7907 token_acc=0.826 temp=1.90 index_temp=2.320 lr=0.80000 batches=1
[continuous] step 350: loss=0.7868 token_acc=0.783 temp=1.80 index_temp=2.255 lr=0.80000 batches=1
[continuous] step 400: loss=0.7818 token_acc=0.826 temp=1.70 index_temp=2.179 lr=0.80000 batches=1
[continuous] step 450: loss=0.7770 token_acc=0.739 temp=1.60 index_temp=2.094 lr=0.80000 batches=1
[continuous] 

In [ ]:
print()
print('=== CONTINUOUS predictions ===')
for inp, ans in demo_cases[:20]:
    pred = demo_brain.predict_continuous(inp)
    print(inp, '->', repr(pred), ' (target', ans, ') match:', pred == ans)
cont_exact = demo_brain.evaluate_continuous(demo_cases, ratio=False)
cont_ratio = demo_brain.evaluate_continuous(demo_cases, ratio=True)
print('continuous exact-match:', cont_exact, ' ratio:', cont_ratio)
#print('continuous test match:', demo_brain.evaluate_continuous(demo_test, ratio=False))


=== CONTINUOUS predictions ===
cosine-decays alongside  -> 'efempidpxem fed'  (target  temp/index_temp for ) match: False
continuous exact-match: 0.0  ratio: 0.5714285714285714


demo_brain.commit_continuous()
print()
print('=== DISCRETE predictions (after commit_continuous) ===')
for inp, ans in demo_cases[:10]:
    pred = ''.join(demo_brain.compute_input(inp))
    print(inp, '->', repr(pred), ' (target', ans, ') match:', pred == ans)
disc_exact = sum(evaluate_performance(i, a, demo_brain, ratio=False) for i, a in demo_cases) / len(demo_cases)
disc_ratio = sum(evaluate_performance(i, a, demo_brain, ratio=True) for i, a in demo_cases) / len(demo_cases)
print('discrete exact-match:', disc_exact, ' ratio:', disc_ratio)
print('discrete test match:', sum(evaluate_performance(i, a, demo_brain, ratio=False) for i, a in demo_test) / len(demo_test))

## Full council training run (optional, slow)

This reproduces the original script's `if __name__ == '__main__':` block: it trains a 50-brain council with the genetic algorithm (400 generations each) and evaluates it. This can take a long time, so it's guarded behind `RUN_FULL_TRAINING`.


In [ ]:
RUN_FULL_TRAINING = False # set to True to actually run the full council training

In [ ]:
if RUN_FULL_TRAINING:
    # Generate the test cases using the existing generation function
    test_cases = generate_test_cases(amount=5, number='1', indiv=False)
    #random.shuffle(test_cases)

    # Split the dataset
    training_0 = test_cases[:int(0.5 * len(test_cases))]
    training_test = test_cases[int(0.5 * len(test_cases)):]


    # Define council parameters
    COUNCIL_SIZE = 50      # Number of voting brains in the council
    GENERATIONS = 200     # Generations per brain
    POP_SIZE = 100        # Population size per generation

    # Train the council (each brain is trained entirely separately)
    print(f"Initializing Council Training with {COUNCIL_SIZE} brains...")
    council = train_council(
        num_brains=COUNCIL_SIZE,
        training_data=training_0,
        alphabet=alphabet,
        output_format=output,
        generations=GENERATIONS,
        pop_size=POP_SIZE
    )



    # Evaluate the entire council's performance against the testing set
    # Note: evaluate_population_parallel relies on a list of brains, so we wrap the council in a list
    print("\nEvaluating Council performance on test batch...")
    council_fitness, _ = evaluate_population_parallel([council], training_test, False)
    council_fitness_training, _ = evaluate_population_parallel([council], training_0, False)
    print(f"Overall Council Fitness: {council_fitness}, Council training fitness: {council_fitness_training}")